In [6]:
import pandas as pd

from langchain_community.llms import CTransformers
from langchain.schema import Document

from langchain.text_splitter import RecursiveCharacterTextSplitter
#!pip install -U langchain-huggingface
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

### Loading and preparing the vector base and finalpassage

In [7]:
# === STEP 1: load CSV ===
df_train = pd.read_csv("data/train.csv")  
df_train = df_train.dropna(subset=["finalpassage","query","answers"])

df_val = pd.read_csv("data/valid.csv")  
df_val = df_val.dropna(subset=["finalpassage","query","answers"])

df_all = pd.concat([df_train,df_val],ignore_index =True)
df_all["finalpassage"] = df_all["finalpassage"].astype(str)

# === Step 2: Convert 'finalpassage' to docs ===
docs = [
    Document(
        page_content=row["finalpassage"]
    )
    for _, row in df_all.iterrows()
]

# === STEP 3: split docs in chunks ===
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)

split_docs[:1]

# === STEP 4: Generate embeddings ===
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# === STEP 5: Create and save the vector database (FAISS) ===
vector_db = FAISS.from_documents(split_docs, embeddings)
vector_db.save_local("vectorial_faiss_db")

### Loading Llama

In [3]:
from langchain.llms import CTransformers

llm = CTransformers(
    model="models/llama-2-7b-chat.Q4_K_M.gguf",
    model_type="llama",
    config={
        "max_new_tokens": 512,
        "temperature": 0.6,
        "context_length": 2048,
        "gpu_layers": 33,  # Max layers to offload to GPU (auto-manages based on VRAM)
    }
)


In [12]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})  # Retrieve top 3 chunks

In [14]:
def rag_query(llm, retriever, query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    context = "\n".join(doc.page_content for doc in docs)

    prompt = f"""You are a helpful assistant. Use the context below to answer the user's question.

Context:
{context}

Question:
{query}

Answer:"""

    return llm(prompt)


In [ ]:
predictions = []
references = []

for _, row in df_val.iterrows():
    query = row["query"]
    answer = row["answers"]

    prediction = rag_query(llm, retriever, query)
    predictions.append(prediction)
    references.append(answer)


/tmp/ipykernel_6268/984353776.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)
/tmp/ipykernel_6268/984353776.py:15: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return llm(prompt)
